Khai báo thư viện và cấu hình hệ thống

In [1]:
import psycopg2
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Cấu hình 3 Cluster (Chi nhánh)
BRANCH_CONFIGS = [
    {"branch_id": 1, "name": "BRANCH1", "host": "26.51.227.155", "port": 26257, "db": "btl2", "user": "root", "pw": "", "schema": "public"},
    {"branch_id": 2, "name": "BRANCH2", "host": "26.250.61.91",  "port": 26257, "db": "btl2", "user": "root", "pw": "", "schema": "public"},
    {"branch_id": 3, "name": "BRANCH3", "host": "26.101.109.119", "port": 26257, "db": "btl2", "user": "root", "pw": "", "schema": "public"}
]

# Hàm hỗ trợ thực thi SQL chung
def execute_query(config, query, params=None, fetch=False):
    conn = None
    try:
        conn = psycopg2.connect(host=config["host"], port=config["port"], database=config["db"], user=config["user"], password=config["pw"], connect_timeout=5)
        cur = conn.cursor()
        cur.execute(query, params)
        if fetch:
            columns = [desc[0] for desc in cur.description]
            res = cur.fetchall()
            conn.commit()
            return pd.DataFrame(res, columns=columns)
        conn.commit()
        return True
    except Exception as e:
        print(f"Lỗi tại {config['name']}: {e}")
        return False
    finally:
        if conn: conn.close()

def get_node(branch_id):
    return next((c for c in BRANCH_CONFIGS if c["branch_id"] == branch_id), None)

In [2]:
def rollback_req3_data():
    print("🔄 Đang dọn dẹp dữ liệu Demo Requirement 3 (ID >= 4000000)...")
    
    # Danh sách các câu lệnh xóa, chạy theo thứ tự NGƯỢC LẠI để tránh lỗi khóa ngoại (Foreign Key)
    queries = [
        "DELETE FROM ORDER_ITEMS WHERE order_id >= 4000000;",
        "DELETE FROM ORDERS WHERE order_id >= 4000000;",
        "DELETE FROM PRODUCTS_STOCK WHERE product_id >= 4000000;",
        "DELETE FROM EMPLOYEES WHERE employee_id >= 4000000;",
        "DELETE FROM PRODUCTS_INFO WHERE product_id >= 4000000;",
        "DELETE FROM CUSTOMERS WHERE customer_id >= 4000000;",
        
        # Lưu ý: Nếu bảng BRANCHES của bạn dùng ID nhỏ (1, 2, 3), mình sẽ không xóa. 
        # Nếu chi nhánh demo cũng dùng ID >= 4000000 thì bạn bỏ comment dòng dưới:
        # "DELETE FROM BRANCHES WHERE branch_id >= 4000000;", 
        
        "DELETE FROM OPERATION_LOGS;" # Bảng log này sinh ra ở Req 3 nên xóa sạch để reset trạng thái
    ]
    
    for config in BRANCH_CONFIGS:
        conn = None
        try:
            # Kết nối tới từng cluster
            conn = psycopg2.connect(
                host=config["host"], 
                port=config["port"], 
                database=config["db"], 
                user=config["user"], 
                password=config["pw"]
            )
            cur = conn.cursor()
            
            # Thực thi lần lượt các câu lệnh xóa
            for query in queries:
                try:
                    cur.execute(query)
                except Exception as e:
                    # Bỏ qua nếu bảng chưa tồn tại hoặc lỗi nhỏ để không dừng toàn bộ quá trình
                    print(f"  [!] Bỏ qua tại {config['name']}: {query} -> {e}")
                    
            conn.commit()
            print(f"✅ Đã rollback dữ liệu Req 3 thành công trên Cluster: {config['name']}")
            
        except Exception as e:
            print(f"❌ Lỗi kết nối tại {config['name']}: {e}")
            if conn: conn.rollback()
        finally:
            if conn: conn.close()
            
    print("🎉 Dọn dẹp hoàn tất! Hệ thống đã giữ nguyên Req 2 và sẵn sàng chạy lại Demo Req 3.")

# Đặt dòng này ở cell đầu tiên để chạy trước mỗi lần demo
rollback_req3_data()

🔄 Đang dọn dẹp dữ liệu Demo Requirement 3 (ID >= 4000000)...
✅ Đã rollback dữ liệu Req 3 thành công trên Cluster: BRANCH1
✅ Đã rollback dữ liệu Req 3 thành công trên Cluster: BRANCH2
✅ Đã rollback dữ liệu Req 3 thành công trên Cluster: BRANCH3
🎉 Dọn dẹp hoàn tất! Hệ thống đã giữ nguyên Req 2 và sẵn sàng chạy lại Demo Req 3.


Khởi tạo bảng Log Replication

In [3]:
create_log_table_sql = """
CREATE TABLE IF NOT EXISTS OPERATION_LOGS (
    log_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    operation VARCHAR(20) NOT NULL,
    table_name VARCHAR(100) NOT NULL,
    record_key VARCHAR(255) NOT NULL,
    source_cluster VARCHAR(50) NOT NULL,
    payload JSONB,
    status VARCHAR(20) DEFAULT 'PENDING',
    created_at TIMESTAMPTZ DEFAULT current_timestamp
);
"""
print("--- KHỞI TẠO BẢNG OPERATION_LOGS TRÊN 3 CLUSTER ---")
for config in BRANCH_CONFIGS:
    success = execute_query(config, create_log_table_sql)
    print(f"[{config['name']}] Khởi tạo bảng Log: {'THÀNH CÔNG' if success else 'THẤT BẠI'}")

--- KHỞI TẠO BẢNG OPERATION_LOGS TRÊN 3 CLUSTER ---
[BRANCH1] Khởi tạo bảng Log: THÀNH CÔNG
[BRANCH2] Khởi tạo bảng Log: THÀNH CÔNG
[BRANCH3] Khởi tạo bảng Log: THÀNH CÔNG


Xây dựng Replication Service

In [4]:
def log_operation(config, op, table, key, payload):
    sql = "INSERT INTO OPERATION_LOGS (operation, table_name, record_key, source_cluster, payload) VALUES (%s, %s, %s, %s, %s)"
    execute_query(config, sql, (op, table, str(key), config["name"], json.dumps(payload)))

def api_post_replicate():
    print("\n[REPLICATION SERVICE] Đang đồng bộ dữ liệu giữa các Cluster...")
    for source_node in BRANCH_CONFIGS:
        # Lấy các log chưa đồng bộ
        logs_df = execute_query(source_node, "SELECT * FROM OPERATION_LOGS WHERE status = 'PENDING' ORDER BY created_at ASC", fetch=True)
        if not isinstance(logs_df, pd.DataFrame) or logs_df.empty: continue

        for _, log in logs_df.iterrows():
            payload = log['payload']
            targets = [n for n in BRANCH_CONFIGS if n["name"] != source_node["name"]]

            for target in targets:
                # Phục hồi dữ liệu UPSERT (Mô phỏng đơn giản cho demo)
                if log['operation'] == 'INSERT':
                    cols = ', '.join(payload.keys())
                    vals = ', '.join(['%s'] * len(payload))
                    sql = f"INSERT INTO {log['table_name']} ({cols}) VALUES ({vals}) ON CONFLICT DO NOTHING"
                    execute_query(target, sql, tuple(payload.values()))

            # Đánh dấu thành công
            execute_query(source_node, "UPDATE OPERATION_LOGS SET status = 'SUCCESS' WHERE log_id = %s", (log['log_id'],))
            print(f"-> Đã đồng bộ {log['table_name']} (Key: {log['record_key']}) từ {source_node['name']} sang các cluster khác.")

Demo Replication với bảng CUSTOMERS

In [5]:
def api_post_customers(data):
    # 1. Tạo một bản copy của data để thao tác, tránh ảnh hưởng dict gốc
    payload = data.copy()
    
    # 2. Rút tham số branch_id ra khỏi payload để dùng định tuyến
    # Dùng pop() để lấy giá trị ra và xóa luôn key này khỏi payload
    target_branch = payload.pop("branch_id", None) 
    if not target_branch: 
        target_branch = payload.pop("source_branch_id", 1) # Fallback nếu gọi nhầm tên

    node = get_node(target_branch)

    # 3. Insert chính (dùng data gốc hoặc payload đều được vì đã mapping rõ cột)
    sql = "INSERT INTO CUSTOMERS (customer_id, customer_name, gender, age, country, city) VALUES (%s, %s, %s, %s, %s, %s)"
    execute_query(node, sql, (payload["customer_id"], payload["customer_name"], payload["gender"], payload["age"], payload["country"], payload["city"]))

    # 4. Ghi log để replicate (Lúc này biến payload đã SẠCH, không còn dính chữ branch_id)
    log_operation(node, "INSERT", "CUSTOMERS", payload["customer_id"], payload)
    print(f"✅ Đã tạo Customer {payload['customer_id']} tại chính {node['name']}")

# ------ THỰC THI DEMO ------
new_customer = {
    "customer_id": 4000001,
    "customer_name": "Nguyen Van A",
    "gender": "Male",
    "age": 25,
    "country": "Vietnam",
    "city": "Ho Chi Minh",
    "branch_id": 1 # Ghi vào Branch 1
}

api_post_customers(new_customer)

print("\n--- TRƯỚC KHI REPLICATE (Xem Branch 2) ---")
display(execute_query(BRANCH_CONFIGS[1], "SELECT * FROM CUSTOMERS WHERE customer_id = 4000001", fetch=True))

api_post_replicate()

print("\n--- SAU KHI REPLICATE (Xem Branch 2) ---")
display(execute_query(BRANCH_CONFIGS[1], "SELECT * FROM CUSTOMERS WHERE customer_id = 4000001", fetch=True))

✅ Đã tạo Customer 4000001 tại chính BRANCH1

--- TRƯỚC KHI REPLICATE (Xem Branch 2) ---


,customer_id,customer_name,gender,age,country,city



[REPLICATION SERVICE] Đang đồng bộ dữ liệu giữa các Cluster...
-> Đã đồng bộ CUSTOMERS (Key: 4000001) từ BRANCH1 sang các cluster khác.

--- SAU KHI REPLICATE (Xem Branch 2) ---


,customer_id,customer_name,gender,age,country,city
0,4000001,Nguyen Van A,Male,25,Vietnam,Ho Chi Minh


CREATE Phân tán (Transaction ORDERS)

In [6]:
def api_post_orders(data):
    node = get_node(data["branch_id"])
    print(f"\n[API ROUTER] Điều hướng request tạo đơn hàng đến {node['name']}...")

    conn = None
    try:
        conn = psycopg2.connect(host=node["host"], port=node["port"], database=node["db"], user=node["user"], password=node["pw"])
        cur = conn.cursor()

        # 1. Insert ORDERS
        cur.execute("INSERT INTO ORDERS (order_id, customer_id, employee_id, payment_method, total_price_usd) VALUES (%s, %s, %s, %s, %s)",
                    (data["order_id"], data["customer_id"], data["employee_id"], data["payment_method"], data["total_price_usd"]))

        # 2. Insert ORDER_ITEMS & Trừ tồn kho PRODUCTS_STOCK
        for item in data["items"]:
            cur.execute("INSERT INTO ORDER_ITEMS (order_id, product_id, quantity, total_price_usd) VALUES (%s, %s, %s, %s)",
                        (data["order_id"], item["product_id"], item["quantity"], item["total_price_usd"]))

            cur.execute("UPDATE PRODUCTS_STOCK SET stock_quantity = stock_quantity - %s WHERE product_id = %s AND branch_id = %s",
                        (item["quantity"], item["product_id"], data["branch_id"]))

        conn.commit()
        print(f"✅ Đã tạo đơn hàng {data['order_id']} thành công tại {node['name']}")
    except Exception as e:
        if conn: conn.rollback()
        print(f"❌ Lỗi tạo đơn hàng: {e}")
    finally:
        if conn: conn.close()

# ------ THỰC THI DEMO ------
node_b2 = get_node(2)

# 1. Tạo nhân viên giả để tránh lỗi Khóa ngoại (Foreign Key)
setup_employee_sql = """
INSERT INTO EMPLOYEES (employee_id, first_name, last_name, branch_id) 
VALUES (5000001, 'Nhân Viên', 'Demo', 2) 
ON CONFLICT (employee_id) DO NOTHING;
"""
execute_query(node_b2, setup_employee_sql)
print("✅ Đã chuẩn bị dữ liệu Nhân viên (ID: 5000001) tại BRANCH2")

# 2. [BỔ SUNG] Tạo sản phẩm giả để tránh lỗi Khóa ngoại cho bảng ORDER_ITEMS
# Sử dụng các cột cơ bản (product_id, product_name) dựa theo cấu trúc database của bạn
setup_product_sql = """
INSERT INTO PRODUCTS_INFO (product_id, product_name, category, brand) 
VALUES (4000001, 'Sản Phẩm Demo', 'Demo', 'Demo') 
ON CONFLICT (product_id) DO NOTHING;
"""
execute_query(node_b2, setup_product_sql)
print("✅ Đã chuẩn bị dữ liệu Sản phẩm (ID: 4000001) tại BRANCH2")

# 3. Khởi tạo đơn hàng (Giữ nguyên như cũ)
new_order = {
    "order_id": 5000001,
    "customer_id": 4000001,
    "employee_id": 5000001, # Đã có
    "branch_id": 2, 
    "payment_method": "Cash",
    "total_price_usd": 704.99,
    "items": [{"product_id": 4000001, "quantity": 1, "total_price_usd": 699.99}] # Đã có
}

# 4. Thực thi
api_post_orders(new_order)

✅ Đã chuẩn bị dữ liệu Nhân viên (ID: 5000001) tại BRANCH2
✅ Đã chuẩn bị dữ liệu Sản phẩm (ID: 4000001) tại BRANCH2

[API ROUTER] Điều hướng request tạo đơn hàng đến BRANCH2...
✅ Đã tạo đơn hàng 5000001 thành công tại BRANCH2


READ Trong suốt (Transparent Read)

In [7]:
def api_get_order(order_id):
    print(f"\n[API ROUTER] Tìm kiếm đơn hàng {order_id} trong suốt trên toàn hệ thống...")
    
    for config in BRANCH_CONFIGS:
        df = execute_query(config, "SELECT * FROM ORDERS WHERE order_id = %s", (order_id,), fetch=True)
        
        # Kiểm tra: Nếu kết nối thành công (trả về DataFrame) và có dữ liệu (không rỗng)
        if isinstance(df, pd.DataFrame) and not df.empty:
            print(f"✅ Tìm thấy tại {config['name']}!")
            return df
            
        # Nếu node này bị sập (trả về False) hoặc bảng rỗng, vòng lặp tự động chuyển sang node tiếp theo
        
    print("❌ Không tìm thấy đơn hàng trên toàn hệ thống.")
    return pd.DataFrame()

# ------ THỰC THI DEMO ------
display(api_get_order(5000001))


[API ROUTER] Tìm kiếm đơn hàng 5000001 trong suốt trên toàn hệ thống...
✅ Tìm thấy tại BRANCH2!


,order_id,order_date,customer_id,employee_id,branch_id,payment_method,tax_usd,total_price_usd
0,5000001,2026-05-29 02:43:29.500235+00:00,4000001,5000001,None,Cash,None,704.99


UPDATE Trong suốt

In [8]:
def api_put_order(order_id, new_payment_method):
    print(f"\n[API ROUTER] Cập nhật đơn hàng {order_id} trong suốt trên toàn hệ thống...")
    
    for config in BRANCH_CONFIGS:
        check_df = execute_query(config, "SELECT order_id FROM ORDERS WHERE order_id = %s", (order_id,), fetch=True)
        
        if isinstance(check_df, pd.DataFrame) and not check_df.empty:
            print(f"🔍 Tìm thấy tại {config['name']}, tiến hành cập nhật...")
            sql = "UPDATE ORDERS SET payment_method = %s WHERE order_id = %s"
            success = execute_query(config, sql, (new_payment_method, order_id))
            
            if success:
                print(f"✅ Đã cập nhật phương thức thanh toán thành '{new_payment_method}'.")
            return
            
    print("❌ Không tìm thấy đơn hàng để cập nhật.")

# ------ THỰC THI DEMO ------
api_put_order(5000001, "Banking")


[API ROUTER] Cập nhật đơn hàng 5000001 trong suốt trên toàn hệ thống...
🔍 Tìm thấy tại BRANCH2, tiến hành cập nhật...
✅ Đã cập nhật phương thức thanh toán thành 'Banking'.


DELETE Trong suốt

In [9]:
def api_delete_order(order_id):
    print(f"\n[API ROUTER] Xóa đơn hàng {order_id} trong suốt trên toàn hệ thống...")
    
    for config in BRANCH_CONFIGS:
        # 1. Tìm xem đơn hàng đang nằm ở đâu
        check_df = execute_query(config, "SELECT order_id FROM ORDERS WHERE order_id = %s", (order_id,), fetch=True)
        
        if isinstance(check_df, pd.DataFrame) and not check_df.empty:
            print(f"🔍 Đã tìm thấy đơn hàng tại {config['name']}, tiến hành xóa...")
            conn = None
            try:
                conn = psycopg2.connect(host=config["host"], port=config["port"], database=config["db"], user=config["user"], password=config["pw"])
                cur = conn.cursor()
                
                # 2. Xóa Items trước (ràng buộc khóa ngoại), xóa Order sau
                cur.execute("DELETE FROM ORDER_ITEMS WHERE order_id = %s", (order_id,))
                cur.execute("DELETE FROM ORDERS WHERE order_id = %s", (order_id,))
                conn.commit()
                print("✅ Xóa thành công, đảm bảo toàn vẹn dữ liệu.")
                return # Xóa xong thì thoát luôn vòng lặp
            except Exception as e:
                if conn: conn.rollback()
                print(f"❌ Lỗi khi xóa: {e}")
            finally:
                if conn: conn.close()
                
    print("❌ Không tìm thấy đơn hàng để xóa.")

# ------ THỰC THI DEMO ------
# Người dùng không cần biết đơn hàng ở nhánh nào, chỉ cần truyền ID
api_delete_order(5000001)


[API ROUTER] Xóa đơn hàng 5000001 trong suốt trên toàn hệ thống...
🔍 Đã tìm thấy đơn hàng tại BRANCH2, tiến hành xóa...
✅ Xóa thành công, đảm bảo toàn vẹn dữ liệu.


Kiểm tra Audit Log Hệ thống

In [10]:
def api_get_logs():
    print("\n--- OPERATION LOGS TỪ BRANCH 1 ---")
    df = execute_query(BRANCH_CONFIGS[0], "SELECT operation, table_name, record_key, status, created_at FROM OPERATION_LOGS ORDER BY created_at DESC", fetch=True)
    display(df)

api_get_logs()


--- OPERATION LOGS TỪ BRANCH 1 ---


,operation,table_name,record_key,status,created_at
0,INSERT,CUSTOMERS,4000001,SUCCESS,2026-05-29 02:43:24.755254+00:00
